# YOLO11 Multi-Class Equipment & Nameplate Detector -> posify backend

Self-contained training notebook (`nameplate`, `pump`, `valve`, `motor`). Select **Runtime -> Change runtime type -> T4 GPU**, then click **Runtime -> Run all**.

In [ ]:
from pathlib import Path
ROOT = Path("/content/posify_ml_starter")
ROOT.mkdir(parents=True, exist_ok=True)
FILES = {'requirements-colab.txt': "# Keep Colab's preinstalled CUDA-enabled torch/torchvision.\nultralytics==8.3.221\nonnx>=1.18.0\nonnxruntime>=1.31.1,<2\nPyYAML>=6.0,<7\nPillow>=10,<13\n", 'validate_dataset.py': '"""Validate YOLO boxes and detect exact image/group leakage before training."""\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\n\nEXTENSIONS = {\'.jpg\', \'.jpeg\', \'.png\', \'.bmp\', \'.webp\'}\n\n\ndef inspect_dataset(config_path, groups_path=None):\n    import yaml\n    from PIL import Image\n    config_path = Path(config_path).resolve()\n    config = yaml.safe_load(config_path.read_text())\n    if not isinstance(config, dict) or \'download\' in config:\n        raise ValueError(\'Use a plain local dataset YAML without a download directive\')\n    names = config.get(\'names\')\n    if isinstance(names, dict):\n        if sorted(names) != list(range(len(names))):\n            raise ValueError(\'Class IDs must be contiguous integers starting at zero\')\n        names = [names[i] for i in range(len(names))]\n    if not isinstance(names, list) or not names or len(set(names)) != len(names) or any(not isinstance(n, str) or not n.strip() for n in names):\n        raise ValueError(\'names must contain unique non-empty class labels\')\n    root = Path(config.get(\'path\') or config_path.parent)\n    if not root.is_absolute(): root = config_path.parent / root\n    root = root.resolve()\n    image_rows, counts, errors, hashes, all_paths = [], {}, [], {}, set()\n    normalized = {\'path\': str(root), \'names\': names}\n    for split in [\'train\', \'val\', \'test\']:\n        value = config.get(split)\n        if not value:\n            if split != \'test\': errors.append(f\'Missing {split} split\')\n            continue\n        if not isinstance(value, str) or \'://\' in value:\n            raise ValueError(\'This starter expects local directory paths for each split\')\n        folder = Path(value)\n        folder = (root / folder).resolve() if not folder.is_absolute() else folder.resolve()\n        if not folder.is_dir():\n            errors.append(f\'{split} directory does not exist: {folder}. Fix paths in data.yaml.\')\n            continue\n        normalized[split] = str(folder)\n        images = sorted(p for p in folder.rglob(\'*\') if p.suffix.lower() in EXTENSIONS)\n        counts[split] = {\'images\': len(images), \'objects\': [0]*len(names), \'negatives\': 0}\n        if not images: errors.append(f\'{split} has no images\')\n        for image in images:\n            resolved = str(image.resolve())\n            if resolved in all_paths: errors.append(f\'Image path appears in multiple splits: {image}\')\n            all_paths.add(resolved)\n            try:\n                with Image.open(image) as opened: opened.verify()\n            except Exception:\n                errors.append(f\'Cannot decode image: {image}\')\n                continue\n            digest = hashlib.sha256(image.read_bytes()).hexdigest()\n            if digest in hashes:\n                errors.append(f\'Duplicate image bytes: {image} and {hashes[digest][1]}\')\n            hashes[digest] = (split, str(image))\n            parts = list(image.parts)\n            positions = [i for i, part in enumerate(parts[:-1]) if part == \'images\']\n            if not positions:\n                errors.append(f\'Image path must contain an images directory: {image}\')\n                continue\n            parts[positions[-1]] = \'labels\'\n            label = Path(*parts).with_suffix(\'.txt\')\n            if not label.is_file():\n                errors.append(f\'Missing label file: {label}. Use an empty TXT for an intentional negative.\')\n                continue\n            lines = [line for line in label.read_text().splitlines() if line.strip()]\n            if not lines: counts[split][\'negatives\'] += 1\n            for number, line in enumerate(lines, 1):\n                try:\n                    fields = line.split()\n                    if len(fields) != 5: raise ValueError(\'Expected class xc yc width height\')\n                    cls = int(fields[0]); x,y,w,h = map(float, fields[1:])\n                    if not 0 <= cls < len(names): raise ValueError(\'Unknown class ID\')\n                    if not all(math.isfinite(v) for v in (x,y,w,h)): raise ValueError(\'Nonfinite box\')\n                    if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1): raise ValueError(\'Box is not normalized\')\n                    if min(x-w/2,y-h/2) < -1e-5 or max(x+w/2,y+h/2) > 1+1e-5: raise ValueError(\'Box crosses image boundary\')\n                    counts[split][\'objects\'][cls] += 1\n                except ValueError as exc:\n                    errors.append(f\'{label}:{number}: {exc}\')\n            image_rows.append({\'path\': str(image), \'split\': split, \'sha256\': digest})\n    notes = [\'Near-duplicates are not detected by byte hashes. Split by physical asset/capture session before augmentation.\']\n    if groups_path:\n        group_map = {}\n        with Path(groups_path).open(newline=\'\') as stream:\n            reader = csv.DictReader(stream)\n            if not {\'image\', \'group_id\'} <= set(reader.fieldnames or []):\n                raise ValueError(\'Groups CSV requires image,group_id columns; paths are relative to dataset root\')\n            for row in reader:\n                key = str((root / row[\'image\']).resolve())\n                if key in group_map or not row[\'group_id\'].strip(): errors.append(f\'Duplicate/empty grouping entry: {key}\')\n                group_map[key] = row[\'group_id\']\n        seen = {}\n        for image in image_rows:\n            group = group_map.get(image[\'path\'])\n            if not group:\n                errors.append(f\'Missing group for {image["path"]}\')\n            elif group in seen and seen[group] != image[\'split\']:\n                errors.append(f\'Group {group} crosses {seen[group]} and {image["split"]}\')\n            else: seen[group] = image[\'split\']\n    else:\n        notes.append(\'No groups CSV supplied: asset/session leakage has NOT been checked.\')\n    if \'train\' in counts and any(n == 0 for n in counts[\'train\'][\'objects\']): errors.append(\'At least one class has no training boxes\')\n    if errors: raise ValueError(\'\\n\'.join(errors[:50]))\n    pairs = []\n    for image in image_rows:\n        parts = list(Path(image[\'path\']).parts); positions = [i for i,p in enumerate(parts[:-1]) if p == \'images\']\n        parts[positions[-1]] = \'labels\'\n        label_hash = hashlib.sha256(Path(*parts).with_suffix(\'.txt\').read_bytes()).hexdigest()\n        pairs.append((image[\'split\'], image[\'sha256\'], label_hash))\n    fingerprint = hashlib.sha256(json.dumps({\'pairs\': sorted(pairs), \'names\': names},sort_keys=True).encode()).hexdigest()\n    return normalized, {\'classes\': names, \'splits\': counts, \'fingerprint\': fingerprint, \'notes\': notes}\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--data\', required=True)\n    parser.add_argument(\'--groups\')\n    parser.add_argument(\'--report\', default=\'dataset_report.json\')\n    args = parser.parse_args()\n    try: _, report = inspect_dataset(args.data, args.groups)\n    except ValueError as exc: parser.exit(2, str(exc)+\'\\n\')\n    Path(args.report).write_text(json.dumps(report, indent=2))\n    print(json.dumps(report, indent=2))\n\n\nif __name__ == \'__main__\': main()\n', 'prepare_multiclass.py': '"""Prepare, remap, and merge YOLO datasets into a validated multi-class layout.\n\nSupports:\n1. Canonical class ordering (e.g. nameplate, pump, valve, motor).\n2. Remapping source-specific class names/IDs (e.g. \'0\' -> \'pump\', \'open\'/\'close\' -> \'valve\').\n3. Merging annotations when the same image (by SHA-256) is annotated across sources.\n4. Deterministic creation of missing val/test splits without cross-split leakage.\n"""\nimport argparse\nimport hashlib\nimport json\nimport shutil\nfrom pathlib import Path\n\nfrom validate_dataset import EXTENSIONS, inspect_dataset\n\n\ndef _load_names(data_cfg):\n    names = data_cfg.get(\'names\')\n    if isinstance(names, dict):\n        if sorted(int(k) for k in names) != list(range(len(names))):\n            raise ValueError(\'Source class IDs must be contiguous integers starting at zero\')\n        return [str(names[i]) if i in names else str(names[str(i)]) for i in range(len(names))]\n    if isinstance(names, list) and names:\n        return [str(n) for n in names]\n    raise ValueError(\'Source data.yaml must define a non-empty names list or mapping\')\n\n\ndef _find_split_dirs(root, data_cfg):\n    found = {}\n    for canonical, candidates in [(\'train\', [\'train\']), (\'val\', [\'val\', \'valid\']), (\'test\', [\'test\'])]:\n        configured = None\n        for key in candidates:\n            if data_cfg.get(key):\n                configured = Path(data_cfg[key])\n                break\n        if configured is not None:\n            folder = (root / configured).resolve() if not configured.is_absolute() else configured.resolve()\n            if not folder.is_dir():\n                for key in candidates:\n                    alt = (root / key / \'images\').resolve()\n                    if alt.is_dir():\n                        folder = alt\n                        break\n            if folder.is_dir():\n                found[canonical] = folder\n                continue\n        for key in candidates:\n            alt = (root / key / \'images\').resolve()\n            if alt.is_dir():\n                found[canonical] = alt\n                break\n    if \'train\' not in found:\n        raise ValueError(f\'No train images folder found under {root}\')\n    return found\n\n\ndef _label_path_for(image_path):\n    parts = list(image_path.parts)\n    positions = [i for i, part in enumerate(parts[:-1]) if part == \'images\']\n    if not positions:\n        raise ValueError(f\'Image path must contain an images directory: {image_path}\')\n    parts[positions[-1]] = \'labels\'\n    return Path(*parts).with_suffix(\'.txt\')\n\n\ndef _box_iou(a, b):\n    ax1, ay1, ax2, ay2 = a[0] - a[2] / 2, a[1] - a[3] / 2, a[0] + a[2] / 2, a[1] + a[3] / 2\n    bx1, by1, bx2, by2 = b[0] - b[2] / 2, b[1] - b[3] / 2, b[0] + b[2] / 2, b[1] + b[3] / 2\n    inter = max(0.0, min(ax2, bx2) - max(ax1, bx1)) * max(0.0, min(ay2, by2) - max(ay1, by1))\n    union = a[2] * a[3] + b[2] * b[3] - inter\n    return inter / union if union > 0 else 0.0\n\n\ndef _assign_missing_splits(records, need_val, need_test):\n    """Deterministically partition train records by SHA-256 into train/val/test when splits are absent."""\n    train_items = sorted([r for r in records.values() if r[\'split\'] == \'train\'], key=lambda r: r[\'sha256\'])\n    n = len(train_items)\n    if (need_val or need_test) and n < 3:\n        raise ValueError(\'Need at least 3 training images to carve out missing val/test splits\')\n    test_count = max(1, round(n * 0.10)) if need_test else 0\n    val_count = max(1, round(n * 0.15)) if need_val else 0\n    if test_count + val_count >= n:\n        raise ValueError(\'Not enough training images to allocate distinct train, val, and test splits\')\n    for idx, item in enumerate(train_items):\n        if idx < test_count:\n            item[\'split\'] = \'test\'\n        elif idx < test_count + val_count:\n            item[\'split\'] = \'val\'\n\n\ndef prepare_multiclass(sources, target_classes, output_dir, ensure_splits=True):\n    """Merge and remap one or more YOLO datasets into output_dir.\n\n    sources: list of dicts:\n      - path: path to source dataset root or data.yaml\n      - class_map: optional dict mapping source class name (or stringified ID) -> target class name (or None to drop)\n      - prefix: optional filename prefix (defaults to s0, s1, ...)\n    target_classes: ordered list of unique target class names\n    """\n    import yaml\n    if not isinstance(target_classes, list) or not target_classes or len(set(target_classes)) != len(target_classes):\n        raise ValueError(\'target_classes must be a non-empty list of unique class names\')\n    target_index = {name: idx for idx, name in enumerate(target_classes)}\n    out = Path(output_dir).resolve()\n    if out.exists() and any(out.iterdir()):\n        raise ValueError(f\'Output directory must be empty: {out}\')\n\n    records = {}  # sha256 -> {\'split\': str, \'path\': Path, \'name\': str, \'boxes\': list[(cls_id, x, y, w, h)], \'sources\': set}\n    source_summaries = []\n    has_val = False\n    has_test = False\n\n    for src_idx, src in enumerate(sources):\n        src_path = Path(src[\'path\']).resolve()\n        yaml_path = src_path if src_path.is_file() else src_path / \'data.yaml\'\n        if not yaml_path.is_file():\n            raise ValueError(f\'Missing data.yaml at {yaml_path}\')\n        root = yaml_path.parent\n        cfg = yaml.safe_load(yaml_path.read_text())\n        if not isinstance(cfg, dict) or \'download\' in cfg:\n            raise ValueError(f\'Invalid or unsafe data.yaml in {yaml_path}\')\n        src_names = _load_names(cfg)\n        raw_map = src.get(\'class_map\') or {name: name for name in src_names}\n        resolved_map = {}\n        for cls_id, cls_name in enumerate(src_names):\n            mapped = raw_map.get(cls_name, raw_map.get(str(cls_id), cls_name if not src.get(\'class_map\') else None))\n            if mapped is not None:\n                if mapped not in target_index:\n                    raise ValueError(f\'Mapped class {mapped!r} from {yaml_path} is not in target_classes {target_classes}\')\n                resolved_map[cls_id] = target_index[mapped]\n        if not resolved_map:\n            raise ValueError(f\'Source {yaml_path} maps no classes into target_classes\')\n\n        split_dirs = _find_split_dirs(root, cfg)\n        if \'val\' in split_dirs:\n            has_val = True\n        if \'test\' in split_dirs:\n            has_test = True\n\n        prefix = src.get(\'prefix\') or f\'src{src_idx}\'\n        src_counts = {\'source\': str(root), \'source_classes\': src_names, \'images_read\': 0, \'boxes_kept\': 0}\n\n        for split, folder in split_dirs.items():\n            images = sorted(p for p in folder.rglob(\'*\') if p.suffix.lower() in EXTENSIONS)\n            for image_path in images:\n                label_path = _label_path_for(image_path)\n                if not label_path.is_file():\n                    raise ValueError(f\'Missing label file: {label_path}\')\n                digest = hashlib.sha256(image_path.read_bytes()).hexdigest()\n                parsed_boxes = []\n                for line_no, line in enumerate(label_path.read_text().splitlines(), 1):\n                    if not line.strip():\n                        continue\n                    parts = line.split()\n                    if len(parts) != 5:\n                        raise ValueError(f\'{label_path}:{line_no}: expected 5 fields\')\n                    src_cls = int(parts[0])\n                    if not 0 <= src_cls < len(src_names):\n                        raise ValueError(f\'{label_path}:{line_no}: unknown source class {src_cls}\')\n                    if src_cls not in resolved_map:\n                        continue\n                    dst_cls = resolved_map[src_cls]\n                    coords = tuple(float(v) for v in parts[1:])\n                    parsed_boxes.append((dst_cls, *coords))\n\n                src_counts[\'images_read\'] += 1\n                src_counts[\'boxes_kept\'] += len(parsed_boxes)\n\n                if digest in records:\n                    existing = records[digest]\n                    if existing[\'split\'] != split:\n                        raise ValueError(\n                            f\'Cross-split duplicate image (sha256={digest[:12]}): \'\n                            f\'{existing["path"]} ({existing["split"]}) vs {image_path} ({split})\'\n                        )\n                    for box in parsed_boxes:\n                        if not any(b[0] == box[0] and _box_iou(b[1:], box[1:]) > 0.95 for b in existing[\'boxes\']):\n                            existing[\'boxes\'].append(box)\n                    existing[\'sources\'].add(prefix)\n                else:\n                    records[digest] = {\n                        \'sha256\': digest,\n                        \'split\': split,\n                        \'path\': image_path,\n                        \'name\': f\'{prefix}_{image_path.stem}_{digest[:8]}{image_path.suffix.lower()}\',\n                        \'boxes\': parsed_boxes,\n                        \'sources\': {prefix},\n                    }\n        source_summaries.append(src_counts)\n\n    if ensure_splits and (not has_val or not has_test):\n        _assign_missing_splits(records, need_val=not has_val, need_test=not has_test)\n\n    out.mkdir(parents=True, exist_ok=True)\n    for split in [\'train\', \'valid\', \'test\']:\n        (out / split / \'images\').mkdir(parents=True, exist_ok=True)\n        (out / split / \'labels\').mkdir(parents=True, exist_ok=True)\n\n    for item in sorted(records.values(), key=lambda r: (r[\'split\'], r[\'name\'])):\n        folder_name = \'valid\' if item[\'split\'] == \'val\' else item[\'split\']\n        dst_img = out / folder_name / \'images\' / item[\'name\']\n        dst_lbl = out / folder_name / \'labels\' / Path(item[\'name\']).with_suffix(\'.txt\')\n        shutil.copy2(item[\'path\'], dst_img)\n        lines = [f\'{cls_id} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\' for cls_id, x, y, w, h in item[\'boxes\']]\n        dst_lbl.write_text(\'\\n\'.join(lines) + (\'\\n\' if lines else \'\'))\n\n    data_yaml = {\n        \'path\': str(out),\n        \'train\': \'train/images\',\n        \'val\': \'valid/images\',\n        \'test\': \'test/images\',\n        \'names\': {i: name for i, name in enumerate(target_classes)},\n    }\n    yaml_out = out / \'data.yaml\'\n    yaml_out.write_text(yaml.safe_dump(data_yaml, sort_keys=False))\n    _, validation_report = inspect_dataset(yaml_out)\n    validation_report[\'sources\'] = source_summaries\n    (out / \'prepare_report.json\').write_text(json.dumps(validation_report, indent=2))\n    return yaml_out, validation_report\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--spec\', help=\'JSON file with {"target_classes": [...], "sources": [...]}\')\n    parser.add_argument(\'--source\', action=\'append\', default=[],\n                        help=\'Dataset directory or data.yaml (can be passed multiple times)\')\n    parser.add_argument(\'--classes\', help=\'Comma-separated canonical target classes, e.g. nameplate,pump,valve,motor\')\n    parser.add_argument(\'--remap\', action=\'append\', default=[],\n                        help=\'Optional class mapping old=new (applies to all --source arguments unless --spec is used)\')\n    parser.add_argument(\'--output\', required=True, help=\'Empty output directory for prepared dataset\')\n    parser.add_argument(\'--no-ensure-splits\', action=\'store_true\',\n                        help=\'Do not automatically carve out missing val/test splits from train\')\n    args = parser.parse_args()\n\n    if args.spec:\n        spec = json.loads(Path(args.spec).read_text())\n        target_classes = spec[\'target_classes\']\n        sources = spec[\'sources\']\n    else:\n        if not args.source or not args.classes:\n            parser.error(\'Provide either --spec or both --source and --classes\')\n        target_classes = [c.strip() for c in args.classes.split(\',\') if c.strip()]\n        remap = {}\n        for item in args.remap:\n            if \'=\' not in item:\n                parser.error(f\'Invalid --remap {item!r}; expected old=new\')\n            k, v = item.split(\'=\', 1)\n            remap[k.strip()] = v.strip() or None\n        sources = [{\'path\': s, \'class_map\': remap or None} for s in args.source]\n\n    yaml_out, report = prepare_multiclass(\n        sources=sources,\n        target_classes=target_classes,\n        output_dir=args.output,\n        ensure_splits=not args.no_ensure_splits,\n    )\n    print(json.dumps(report, indent=2))\n    print(\'Prepared multi-class dataset config:\', yaml_out)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'build_colab_dataset.py': '"""Download public industrial imagery and build a validated 4-class YOLO dataset.\n\nClasses:\n  0: nameplate\n  1: pump\n  2: valve\n  3: motor\n\nWorks in Google Colab without private API keys:\n- Attempts to download pinned CC-licensed Wikimedia Commons nameplate/equipment photos.\n- Generates group-isolated train/valid/test industrial equipment + nameplate scenes\n  with exact YOLO bounding boxes (incorporating real photo textures when available).\n- Validates the resulting dataset with validate_dataset.inspect_dataset().\n"""\nimport argparse\nimport hashlib\nimport io\nimport json\nimport random\nimport shutil\nimport urllib.parse\nimport urllib.request\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image, ImageDraw, ImageFilter, ImageOps\n\nfrom validate_dataset import inspect_dataset\n\nCLASSES = [\'nameplate\', \'pump\', \'valve\', \'motor\']\n\nCOMMONS_CATEGORIES = {\n    \'nameplate\': [\'Category:Nameplates\', \'Category:Machine_nameplates\'],\n    \'pump\': [\'Category:Centrifugal_pumps\', \'Category:Pumps\'],\n    \'valve\': [\'Category:Industrial_valves\', \'Category:Gate_valves\', \'Category:Ball_valves\'],\n    \'motor\': [\'Category:Electric_motors\', \'Category:Induction_motors\'],\n}\n\nPUBLIC_COMMONS_SOURCES = [\n    {\n        \'id\': \'electrolux\',\n        \'url\': \'https://upload.wikimedia.org/wikipedia/commons/4/4c/Motor_Nameplate.jpg\',\n        \'kind\': \'nameplate\',\n    },\n    {\n        \'id\': \'current-transformer\',\n        \'url\': \'https://upload.wikimedia.org/wikipedia/commons/1/19/Nameplate_of_a_CT.jpg\',\n        \'kind\': \'nameplate\',\n    },\n    {\n        \'id\': \'malmbergs\',\n        \'url\': \'https://upload.wikimedia.org/wikipedia/commons/f/f9/Motor_nameplate_Malmbergs.jpg\',\n        \'kind\': \'nameplate\',\n    },\n    {\n        \'id\': \'westinghouse\',\n        \'url\': \'https://upload.wikimedia.org/wikipedia/commons/a/af/Nameplate_for_one_of_Fireboat_Firefighter_%27s_600V_DC_Marine_Motors.jpg\',\n        \'kind\': \'nameplate\',\n    },\n]\n\n\ndef _fetch_real_plates(cache_dir, timeout=8):\n    """Best-effort download of public CC nameplate & equipment images from Wikimedia Commons."""\n    bank = {cls: [] for cls in CLASSES}\n    cache_dir = Path(cache_dir)\n    cache_dir.mkdir(parents=True, exist_ok=True)\n    headers = {\'User-Agent\': \'PosifyMLStarter/2.0 (https://github.com/tanish19078/posify)\'}\n    for item in PUBLIC_COMMONS_SOURCES:\n        target = cache_dir / f"{item[\'id\']}.jpg"\n        try:\n            if not target.is_file():\n                req = urllib.request.Request(item[\'url\'], headers=headers)\n                with urllib.request.urlopen(req, timeout=timeout) as resp:\n                    target.write_bytes(resp.read())\n            with Image.open(target) as img:\n                rgb = ImageOps.exif_transpose(img).convert(\'RGB\')\n                rgb.thumbnail((480, 320), Image.Resampling.LANCZOS)\n                bank[\'nameplate\'].append(rgb.copy())\n        except Exception:\n            continue\n\n    for cls_name, categories in COMMONS_CATEGORIES.items():\n        for cat in categories:\n            if len(bank[cls_name]) >= 18:\n                break\n            params = urllib.parse.urlencode({\n                \'action\': \'query\',\n                \'generator\': \'categorymembers\',\n                \'gcmtitle\': cat,\n                \'gcmtype\': \'file\',\n                \'gcmlimit\': \'20\',\n                \'prop\': \'imageinfo\',\n                \'iiprop\': \'url|mime\',\n                \'iiurlwidth\': \'640\',\n                \'format\': \'json\',\n            })\n            api_url = f\'https://commons.wikimedia.org/w/api.php?{params}\'\n            try:\n                req = urllib.request.Request(api_url, headers=headers)\n                with urllib.request.urlopen(req, timeout=timeout) as resp:\n                    payload = json.loads(resp.read().decode(\'utf-8\'))\n                pages = (payload.get(\'query\') or {}).get(\'pages\') or {}\n                for page_id, page in sorted(pages.items()):\n                    if len(bank[cls_name]) >= 18:\n                        break\n                    infos = page.get(\'imageinfo\') or []\n                    if not infos:\n                        continue\n                    mime = infos[0].get(\'mime\', \'\')\n                    thumb_url = infos[0].get(\'thumburl\') or infos[0].get(\'url\')\n                    if mime not in {\'image/jpeg\', \'image/png\'} or not thumb_url:\n                        continue\n                    img_file = cache_dir / f\'{cls_name}_{page_id}.jpg\'\n                    if not img_file.is_file():\n                        img_req = urllib.request.Request(thumb_url, headers=headers)\n                        with urllib.request.urlopen(img_req, timeout=timeout) as img_resp:\n                            img_file.write_bytes(img_resp.read())\n                    with Image.open(img_file) as opened:\n                        rgb = ImageOps.exif_transpose(opened).convert(\'RGB\')\n                        if min(rgb.size) >= 64:\n                            rgb.thumbnail((480, 480), Image.Resampling.LANCZOS)\n                            bank[cls_name].append(rgb.copy())\n            except Exception:\n                continue\n    return bank\n\n\ndef _draw_background(rng, size=640):\n    base_r = rng.randint(35, 175)\n    base_g = max(25, min(200, base_r + rng.randint(-18, 18)))\n    base_b = max(25, min(210, base_r + rng.randint(-15, 25)))\n    arr = np.zeros((size, size, 3), dtype=np.float32)\n    y_grad = np.linspace(0.85, 1.15, size, dtype=np.float32)[:, None]\n    x_grad = np.linspace(0.92, 1.08, size, dtype=np.float32)[None, :]\n    arr[:, :, 0] = base_r * y_grad * x_grad\n    arr[:, :, 1] = base_g * y_grad * x_grad\n    arr[:, :, 2] = base_b * y_grad * x_grad\n    noise = rng.normal(0, rng.uniform(4.0, 11.0), (size, size, 3)).astype(np.float32)\n    arr = np.clip(arr + noise, 0, 255).astype(np.uint8)\n    img = Image.fromarray(arr, mode=\'RGB\')\n    draw = ImageDraw.Draw(img)\n    # Background structural pipes / panel seams\n    for _ in range(rng.randint(2, 5)):\n        if rng.random() < 0.5:\n            y = rng.randint(20, size - 20)\n            h = rng.randint(4, 16)\n            c = tuple(max(10, min(240, base_r + rng.randint(-35, 35))) for _ in range(3))\n            draw.rectangle([0, y, size, y + h], fill=c)\n        else:\n            x = rng.randint(20, size - 20)\n            w = rng.randint(4, 16)\n            c = tuple(max(10, min(240, base_r + rng.randint(-35, 35))) for _ in range(3))\n            draw.rectangle([x, 0, x + w, size], fill=c)\n    return img\n\n\ndef _get_crops(real_crops, kind):\n    if isinstance(real_crops, dict):\n        return real_crops.get(kind) or []\n    return real_crops if kind == \'nameplate\' and isinstance(real_crops, list) else []\n\n\ndef _draw_nameplate(draw, img, box, rng, real_crops):\n    x1, y1, x2, y2 = [int(round(v)) for v in box]\n    w, h = max(24, x2 - x1), max(18, y2 - y1)\n    plates = _get_crops(real_crops, \'nameplate\')\n    if plates and rng.random() < 0.65:\n        src = plates[rng.randint(0, len(plates) - 1)]\n        patch = src.resize((w, h), Image.Resampling.BILINEAR)\n        if rng.random() < 0.5:\n            patch = ImageOps.autocontrast(patch, cutoff=1)\n        img.paste(patch, (x1, y1))\n        draw.rectangle([x1, y1, x2, y2], outline=(35, 35, 40), width=2)\n        return\n\n    style = rng.choice([\'dark_anodized\', \'brushed_silver\', \'industrial_brass\', \'blue_header\'])\n    if style == \'dark_anodized\':\n        bg, fg, accent = (28, 32, 38), (230, 234, 240), (200, 65, 50)\n    elif style == \'brushed_silver\':\n        bg, fg, accent = (212, 218, 224), (22, 28, 36), (25, 75, 145)\n    elif style == \'industrial_brass\':\n        bg, fg, accent = (195, 168, 98), (30, 24, 15), (140, 35, 25)\n    else:\n        bg, fg, accent = (228, 232, 238), (20, 26, 34), (18, 62, 130)\n\n    draw.rectangle([x1, y1, x2, y2], fill=bg, outline=(55, 60, 68), width=2)\n    draw.rectangle([x1 + 3, y1 + 3, x2 - 3, y2 - 3], outline=fg, width=1)\n    # Corner rivets\n    for rx, ry in [(x1 + 6, y1 + 6), (x2 - 6, y1 + 6), (x1 + 6, y2 - 6), (x2 - 6, y2 - 6)]:\n        draw.ellipse([rx - 2, ry - 2, rx + 2, ry + 2], fill=(120, 125, 132), outline=(40, 40, 45))\n    # Header banner\n    hdr_h = max(6, int(h * 0.22))\n    draw.rectangle([x1 + 5, y1 + 5, x2 - 5, y1 + 5 + hdr_h], fill=accent)\n    draw.rectangle([x1 + 9, y1 + 7, x1 + int(w * 0.55), y1 + 3 + hdr_h], fill=(245, 245, 248))\n    # Technical rating table rows and cells\n    rows = rng.randint(3, 6)\n    top_start = y1 + 8 + hdr_h\n    avail_h = max(12, y2 - 6 - top_start)\n    row_h = max(3, avail_h // (rows + 1))\n    for r in range(rows):\n        ry = top_start + r * row_h\n        if ry + 2 >= y2 - 5:\n            break\n        draw.line([(x1 + 6, ry), (x2 - 6, ry)], fill=fg, width=1)\n        # Simulated text blocks\n        cols = rng.randint(2, 4)\n        col_w = max(8, (w - 16) // cols)\n        for c in range(cols):\n            cx = x1 + 8 + c * col_w\n            tw = rng.randint(max(4, col_w // 3), max(5, col_w - 4))\n            draw.rectangle([cx, ry + 2, min(x2 - 7, cx + tw), min(y2 - 6, ry + max(2, row_h - 2))], fill=fg)\n\n\ndef _draw_motor(draw, img, box, rng, real_crops):\n    x1, y1, x2, y2 = [int(round(v)) for v in box]\n    w, h = x2 - x1, y2 - y1\n    motors = _get_crops(real_crops, \'motor\')\n    if motors and rng.random() < 0.55:\n        src = motors[rng.randint(0, len(motors) - 1)]\n        patch = src.resize((max(32, w), max(32, h)), Image.Resampling.BILINEAR)\n        img.paste(patch, (x1, y1))\n        if w >= 120 and h >= 95 and rng.random() < 0.75:\n            pw = rng.randint(int(w * 0.25), int(w * 0.42))\n            ph = rng.randint(int(h * 0.20), int(h * 0.34))\n            px1 = x1 + rng.randint(14, max(15, w - pw - 14))\n            py1 = y1 + rng.randint(14, max(15, h - ph - 14))\n            plate_box = (px1, py1, px1 + pw, py1 + ph)\n            _draw_nameplate(draw, img, plate_box, rng, real_crops)\n            return [plate_box]\n        return []\n    body_color = rng.choice([(42, 78, 118), (52, 88, 72), (68, 74, 84), (35, 95, 135)])\n    dark_trim = tuple(max(10, c - 30) for c in body_color)\n    light_trim = tuple(min(245, c + 35) for c in body_color)\n\n    # Mounting feet\n    foot_h = max(6, int(h * 0.10))\n    draw.rectangle([x1 + int(w * 0.12), y2 - foot_h, x2 - int(w * 0.18), y2], fill=dark_trim, outline=(25, 25, 30))\n    # Main stator cylinder\n    body_x1, body_y1 = x1 + int(w * 0.08), y1 + int(h * 0.14)\n    body_x2, body_y2 = x2 - int(w * 0.16), y2 - foot_h\n    draw.rounded_rectangle([body_x1, body_y1, body_x2, body_y2], radius=10, fill=body_color, outline=dark_trim, width=3)\n    # Fan cowl (left)\n    draw.rounded_rectangle([x1, body_y1 + 4, body_x1 + 8, body_y2 - 4], radius=8, fill=dark_trim, outline=(20, 20, 25), width=2)\n    # Drive shaft (right)\n    shaft_y1 = (body_y1 + body_y2) // 2 - max(4, int(h * 0.07))\n    shaft_y2 = (body_y1 + body_y2) // 2 + max(4, int(h * 0.07))\n    draw.rectangle([body_x2 - 2, shaft_y1, x2, shaft_y2], fill=(185, 192, 200), outline=(70, 75, 80), width=2)\n    # Cooling fins\n    fin_count = rng.randint(5, 9)\n    for idx in range(fin_count):\n        fy = body_y1 + 8 + idx * max(4, (body_y2 - body_y1 - 16) // fin_count)\n        draw.line([(body_x1 + 10, fy), (body_x2 - 10, fy)], fill=light_trim, width=2)\n    # Top terminal box\n    tbox_x1 = body_x1 + int((body_x2 - body_x1) * 0.28)\n    tbox_x2 = body_x1 + int((body_x2 - body_x1) * 0.68)\n    draw.rectangle([tbox_x1, y1, tbox_x2, body_y1 + 4], fill=dark_trim, outline=(25, 25, 30), width=2)\n\n    # Optional mounted nameplate on motor body\n    if w >= 120 and h >= 95 and rng.random() < 0.75:\n        pw = rng.randint(int(w * 0.25), int(w * 0.42))\n        ph = rng.randint(int(h * 0.20), int(h * 0.34))\n        px1 = body_x1 + rng.randint(12, max(13, (body_x2 - body_x1) - pw - 12))\n        py1 = body_y1 + rng.randint(10, max(11, (body_y2 - body_y1) - ph - 10))\n        plate_box = (px1, py1, px1 + pw, py1 + ph)\n        _draw_nameplate(draw, img, plate_box, rng, real_crops)\n        return [plate_box]\n    return []\n\n\ndef _draw_pump(draw, img, box, rng, real_crops):\n    x1, y1, x2, y2 = [int(round(v)) for v in box]\n    w, h = x2 - x1, y2 - y1\n    pumps = _get_crops(real_crops, \'pump\')\n    if pumps and rng.random() < 0.55:\n        src = pumps[rng.randint(0, len(pumps) - 1)]\n        patch = src.resize((max(32, w), max(32, h)), Image.Resampling.BILINEAR)\n        img.paste(patch, (x1, y1))\n        if w >= 120 and h >= 100 and rng.random() < 0.65:\n            pw = rng.randint(int(w * 0.24), int(w * 0.38))\n            ph = rng.randint(int(h * 0.18), int(h * 0.28))\n            px1 = x1 + (w - pw) // 2\n            py1 = y1 + max(10, h - ph - 14)\n            plate_box = (px1, py1, px1 + pw, py1 + ph)\n            _draw_nameplate(draw, img, plate_box, rng, real_crops)\n            return [plate_box]\n        return []\n    pump_color = rng.choice([(25, 110, 165), (38, 128, 98), (165, 65, 45), (55, 85, 140)])\n    dark = tuple(max(10, c - 35) for c in pump_color)\n    flange_gray = (155, 162, 172)\n\n    # Baseplate\n    base_h = max(8, int(h * 0.11))\n    draw.rectangle([x1 + int(w * 0.05), y2 - base_h, x2 - int(w * 0.05), y2], fill=(60, 65, 72), outline=(30, 32, 36), width=2)\n    # Top discharge neck & flange\n    neck_x1 = x1 + int(w * 0.34)\n    neck_x2 = x1 + int(w * 0.62)\n    draw.rectangle([neck_x1, y1 + 10, neck_x2, y1 + int(h * 0.38)], fill=pump_color, outline=dark, width=2)\n    draw.rectangle([neck_x1 - 8, y1, neck_x2 + 8, y1 + 12], fill=flange_gray, outline=(50, 55, 60), width=2)\n    # Circular volute casing\n    vol_x1, vol_y1 = x1 + int(w * 0.14), y1 + int(h * 0.22)\n    vol_x2, vol_y2 = x2 - int(w * 0.14), y2 - base_h + 2\n    draw.ellipse([vol_x1, vol_y1, vol_x2, vol_y2], fill=pump_color, outline=dark, width=3)\n    # Inner impeller hub ring\n    cx, cy = (vol_x1 + vol_x2) // 2, (vol_y1 + vol_y2) // 2\n    rad = max(10, min(vol_x2 - vol_x1, vol_y2 - vol_y1) // 4)\n    draw.ellipse([cx - rad, cy - rad, cx + rad, cy + rad], fill=flange_gray, outline=dark, width=2)\n    # Suction flange (left)\n    draw.rectangle([x1, cy - rad, vol_x1 + 8, cy + rad], fill=flange_gray, outline=(45, 50, 55), width=2)\n\n    if w >= 120 and h >= 100 and rng.random() < 0.65:\n        pw = rng.randint(int(w * 0.24), int(w * 0.38))\n        ph = rng.randint(int(h * 0.18), int(h * 0.28))\n        px1 = cx - pw // 2\n        py1 = min(y2 - ph - 6, vol_y2 - ph - 8)\n        plate_box = (px1, py1, px1 + pw, py1 + ph)\n        _draw_nameplate(draw, img, plate_box, rng, real_crops)\n        return [plate_box]\n    return []\n\n\ndef _draw_valve(draw, img, box, rng, real_crops):\n    x1, y1, x2, y2 = [int(round(v)) for v in box]\n    w, h = x2 - x1, y2 - y1\n    valves = _get_crops(real_crops, \'valve\')\n    if valves and rng.random() < 0.55:\n        src = valves[rng.randint(0, len(valves) - 1)]\n        patch = src.resize((max(32, w), max(32, h)), Image.Resampling.BILINEAR)\n        img.paste(patch, (x1, y1))\n        if w >= 115 and h >= 105 and rng.random() < 0.55:\n            pw = rng.randint(int(w * 0.25), int(w * 0.40))\n            ph = rng.randint(int(h * 0.16), int(h * 0.24))\n            px1 = x1 + (w - pw) // 2\n            py1 = y1 + (h - ph) // 2\n            plate_box = (px1, py1, px1 + pw, py1 + ph)\n            _draw_nameplate(draw, img, plate_box, rng, real_crops)\n            return [plate_box]\n        return []\n    wheel_color = rng.choice([(195, 45, 40), (215, 135, 25), (40, 105, 175)])\n    body_metal = rng.choice([(135, 142, 152), (95, 112, 128), (155, 145, 125)])\n    dark_metal = tuple(max(20, c - 35) for c in body_metal)\n\n    cx = (x1 + x2) // 2\n    body_y1 = y1 + int(h * 0.52)\n    body_y2 = y2 - int(h * 0.06)\n    # Left and right pipe flanges\n    flange_w = max(8, int(w * 0.11))\n    draw.rectangle([x1, body_y1 - 6, x1 + flange_w, body_y2 + 6], fill=body_metal, outline=dark_metal, width=2)\n    draw.rectangle([x2 - flange_w, body_y1 - 6, x2, body_y2 + 6], fill=body_metal, outline=dark_metal, width=2)\n    # Valve globe/gate body\n    draw.rectangle([x1 + flange_w, body_y1 + 4, x2 - flange_w, body_y2 - 4], fill=body_metal, outline=dark_metal, width=2)\n    draw.ellipse([cx - int(w * 0.22), body_y1 - 2, cx + int(w * 0.22), body_y2 + 2], fill=body_metal, outline=dark_metal, width=3)\n    # Stem & yoke bonnet\n    stem_w = max(6, int(w * 0.10))\n    wheel_h = max(12, int(h * 0.16))\n    draw.rectangle([cx - stem_w, y1 + wheel_h, cx + stem_w, body_y1 + 4], fill=dark_metal, outline=(30, 35, 40), width=2)\n    # Top handwheel\n    draw.ellipse([x1 + int(w * 0.15), y1, x2 - int(w * 0.15), y1 + wheel_h], fill=wheel_color, outline=(35, 20, 20), width=3)\n\n    if w >= 115 and h >= 105 and rng.random() < 0.55:\n        pw = rng.randint(int(w * 0.25), int(w * 0.40))\n        ph = rng.randint(int(h * 0.16), int(h * 0.24))\n        px1 = cx - pw // 2\n        py1 = body_y1 + max(4, (body_y2 - body_y1 - ph) // 2)\n        plate_box = (px1, py1, px1 + pw, py1 + ph)\n        _draw_nameplate(draw, img, plate_box, rng, real_crops)\n        return [plate_box]\n    return []\n\n\ndef _to_yolo(cls_id, box, size=640):\n    x1, y1, x2, y2 = box\n    x1, y1 = max(0.0, min(float(size), float(x1))), max(0.0, min(float(size), float(y1)))\n    x2, y2 = max(0.0, min(float(size), float(x2))), max(0.0, min(float(size), float(y2)))\n    w, h = (x2 - x1) / size, (y2 - y1) / size\n    xc, yc = (x1 + x2) / (2.0 * size), (y1 + y2) / (2.0 * size)\n    return f\'{cls_id} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\'\n\n\ndef build_dataset(output_dir, train_count=80, val_count=20, test_count=20, size=640, fetch_remote=True):\n    import yaml\n    out = Path(output_dir).resolve()\n    if out.exists():\n        shutil.rmtree(out)\n    for split in [\'train\', \'valid\', \'test\']:\n        (out / split / \'images\').mkdir(parents=True, exist_ok=True)\n        (out / split / \'labels\').mkdir(parents=True, exist_ok=True)\n\n    cache_dir = out.parent / \'_public_cache\'\n    real_crops = _fetch_real_plates(cache_dir) if fetch_remote else []\n\n    split_specs = [(\'train\', train_count, 1001), (\'valid\', val_count, 2002), (\'test\', test_count, 3003)]\n    for split, count, seed in split_specs:\n        rng = random.Random(seed)\n        np_rng = np.random.default_rng(seed)\n\n        class _CombinedRng:\n            def randint(self, a, b): return rng.randint(a, b)\n            def uniform(self, a, b): return rng.uniform(a, b)\n            def random(self): return rng.random()\n            def choice(self, seq): return rng.choice(seq)\n            def normal(self, loc, scale, shape): return np_rng.normal(loc, scale, shape)\n\n        crng = _CombinedRng()\n        for idx in range(count):\n            img = _draw_background(crng, size=size)\n            draw = ImageDraw.Draw(img)\n            labels = []\n\n            # Keep ~5% intentional negatives\n            if idx % 20 == 19:\n                pass\n            elif idx % 4 == 0:\n                # Close-up nameplate scene (1 to 2 plates)\n                num_plates = 1 if rng.random() < 0.7 else 2\n                slots = [(60, 80, 560, 500)] if num_plates == 1 else [(40, 50, 310, 270), (290, 310, 590, 560)]\n                for sx1, sy1, sx2, sy2 in slots:\n                    pw = rng.randint(max(120, (sx2 - sx1) // 2), sx2 - sx1)\n                    ph = rng.randint(max(80, (sy2 - sy1) // 2), sy2 - sy1)\n                    px1 = rng.randint(sx1, max(sx1, sx2 - pw))\n                    py1 = rng.randint(sy1, max(sy1, sy2 - ph))\n                    pbox = (px1, py1, px1 + pw, py1 + ph)\n                    _draw_nameplate(draw, img, pbox, crng, real_crops)\n                    labels.append(_to_yolo(0, pbox, size))\n            else:\n                # Multi-object equipment + nameplate scene (1 or 2 equipment items)\n                quadrants = [\n                    (35, 45, 315, 315),\n                    (325, 45, 605, 315),\n                    (35, 325, 315, 595),\n                    (325, 325, 605, 595),\n                ]\n                rng.shuffle(quadrants)\n                num_items = 2 if rng.random() < 0.65 else 1\n                chosen_quads = quadrants[:num_items] if num_items == 2 else [(70, 70, 570, 570)]\n                for q_idx, (qx1, qy1, qx2, qy2) in enumerate(chosen_quads):\n                    eq_cls = ((idx + q_idx) % 3) + 1  # 1: pump, 2: valve, 3: motor\n                    ew = rng.randint(max(130, int((qx2 - qx1) * 0.62)), qx2 - qx1)\n                    eh = rng.randint(max(120, int((qy2 - qy1) * 0.62)), qy2 - qy1)\n                    ex1 = rng.randint(qx1, max(qx1, qx2 - ew))\n                    ey1 = rng.randint(qy1, max(qy1, qy2 - eh))\n                    ebox = (ex1, ey1, ex1 + ew, ey1 + eh)\n                    if eq_cls == 1:\n                        plates = _draw_pump(draw, img, ebox, crng, real_crops)\n                    elif eq_cls == 2:\n                        plates = _draw_valve(draw, img, ebox, crng, real_crops)\n                    else:\n                        plates = _draw_motor(draw, img, ebox, crng, real_crops)\n                    labels.append(_to_yolo(eq_cls, ebox, size))\n                    for pbox in plates:\n                        labels.append(_to_yolo(0, pbox, size))\n\n            if rng.random() < 0.25:\n                img = img.filter(ImageFilter.GaussianBlur(radius=rng.uniform(0.3, 0.8)))\n\n            img_path = out / split / \'images\' / f\'{split}_{idx:04d}.jpg\'\n            lbl_path = out / split / \'labels\' / f\'{split}_{idx:04d}.txt\'\n            img.save(img_path, format=\'JPEG\', quality=94)\n            lbl_path.write_text(\'\\n\'.join(labels) + (\'\\n\' if labels else \'\'))\n\n    data_yaml = {\n        \'path\': str(out),\n        \'train\': \'train/images\',\n        \'val\': \'valid/images\',\n        \'test\': \'test/images\',\n        \'names\': {i: name for i, name in enumerate(CLASSES)},\n    }\n    yaml_path = out / \'data.yaml\'\n    yaml_path.write_text(yaml.safe_dump(data_yaml, sort_keys=False))\n    _, report = inspect_dataset(yaml_path)\n    report[\'real_commons_textures_loaded\'] = (\n        {k: len(v) for k, v in real_crops.items()} if isinstance(real_crops, dict) else len(real_crops)\n    )\n    (out / \'dataset_report.json\').write_text(json.dumps(report, indent=2))\n    return yaml_path, report\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--output\', default=\'/content/dataset\', help=\'Target directory for YOLO dataset\')\n    parser.add_argument(\'--train-count\', type=int, default=80)\n    parser.add_argument(\'--val-count\', type=int, default=20)\n    parser.add_argument(\'--test-count\', type=int, default=20)\n    parser.add_argument(\'--no-remote\', action=\'store_true\', help=\'Skip downloading Wikimedia Commons textures\')\n    args = parser.parse_args()\n    yaml_path, report = build_dataset(\n        output_dir=args.output,\n        train_count=args.train_count,\n        val_count=args.val_count,\n        test_count=args.test_count,\n        fetch_remote=not args.no_remote,\n    )\n    print(json.dumps(report, indent=2))\n    print(\'Dataset ready at:\', yaml_path)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'train_yolo.py': '"""Fine-tune a YOLO11 detection model on a validated local YOLO dataset."""\nimport argparse\nimport json\nfrom pathlib import Path\n\nfrom validate_dataset import inspect_dataset\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--data\', required=True)\n    parser.add_argument(\'--groups\')\n    parser.add_argument(\'--weights\', default=\'yolo11n.pt\')\n    parser.add_argument(\'--epochs\', type=int, default=60)\n    parser.add_argument(\'--batch\', type=int, default=8)\n    parser.add_argument(\'--imgsz\', type=int, default=640)\n    parser.add_argument(\'--device\', default=\'0\')\n    parser.add_argument(\'--project\', default=\'/content/runs\')\n    parser.add_argument(\'--name\', default=\'nameplate_yolo11n\')\n    args = parser.parse_args()\n    if args.epochs < 1 or args.batch < 1 or args.imgsz < 32 or args.imgsz % 32:\n        parser.error(\'Use positive epochs/batch and an image size divisible by 32\')\n    import yaml\n    config, report = inspect_dataset(args.data, args.groups)\n    import torch\n    from ultralytics import YOLO\n    if args.device != \'cpu\' and not torch.cuda.is_available():\n        parser.error(\'Enable a GPU runtime in Colab or pass --device cpu\')\n    target = Path(args.project).resolve() / args.name\n    if target.exists(): parser.error(\'Run directory exists; choose a fresh --name to avoid overwriting artifacts\')\n    prepared = Path(args.project).resolve() / (args.name + \'_inputs\')\n    prepared.mkdir(parents=True, exist_ok=False)\n    normalized = prepared / \'data.yaml\'\n    normalized.write_text(yaml.safe_dump(config, sort_keys=False))\n    (prepared / \'dataset_report.json\').write_text(json.dumps(report, indent=2))\n    model = YOLO(args.weights)  # Official YOLO11 pretrained weights download when absent.\n    if model.task != \'detect\': parser.error(\'Only object detection checkpoints are supported\')\n    model.train(data=str(normalized), epochs=args.epochs, imgsz=args.imgsz, batch=args.batch,\n                device=args.device, project=str(Path(args.project).resolve()), name=args.name,\n                seed=42, deterministic=True, workers=2, patience=15, exist_ok=False,\n                amp=True, hsv_h=.01, hsv_s=.3, hsv_v=.3, degrees=8,\n                translate=.1, scale=.3, fliplr=0, flipud=0, mosaic=.5, close_mosaic=10)\n    # Avoid mirror augmentation for nameplate OCR. No accuracy target is assumed.\n    print(\'Training complete. Inspect:\', target)\n    print(\'Best checkpoint:\', target / \'weights\' / \'best.pt\')\n\n\nif __name__ == \'__main__\': main()\n', 'evaluate_yolo.py': '"""Evaluate a trained detector on validation or a previously untouched test split."""\nimport argparse\nimport json\nfrom pathlib import Path\n\nfrom validate_dataset import inspect_dataset\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--weights\', required=True)\n    parser.add_argument(\'--data\', required=True)\n    parser.add_argument(\'--groups\')\n    parser.add_argument(\'--split\', choices=[\'val\',\'test\'], default=\'test\')\n    parser.add_argument(\'--device\', default=\'0\')\n    parser.add_argument(\'--imgsz\', type=int, default=640)\n    parser.add_argument(\'--output\', default=\'detector_evaluation.json\')\n    args = parser.parse_args()\n    config, report = inspect_dataset(args.data, args.groups)\n    if args.split not in config: parser.error(\'Requested split is absent; prepare a held-out test set first\')\n    import tempfile\n    import yaml\n    from ultralytics import YOLO\n    with tempfile.TemporaryDirectory() as temp:\n        path = Path(temp)/\'data.yaml\'; path.write_text(yaml.safe_dump(config))\n        result = YOLO(args.weights).val(data=str(path), split=args.split, device=args.device,\n                                       imgsz=args.imgsz, batch=8, plots=True)\n    metrics = {k: float(v) for k,v in result.results_dict.items()}\n    per_class = {}\n    box_metrics = getattr(result, \'box\', None)\n    if box_metrics is not None and hasattr(box_metrics, \'ap_class_index\'):\n        for idx_pos, cls_idx in enumerate(box_metrics.ap_class_index):\n            cls_id = int(cls_idx)\n            cls_name = report[\'classes\'][cls_id] if 0 <= cls_id < len(report[\'classes\']) else str(cls_id)\n            p, r, ap50, ap = box_metrics.class_result(idx_pos)\n            per_class[cls_name] = {\n                \'class_id\': cls_id,\n                \'precision\': float(p),\n                \'recall\': float(r),\n                \'map50\': float(ap50),\n                \'map50_95\': float(ap),\n            }\n    output = {\'split\': args.split, \'dataset\': report, \'metrics\': metrics, \'per_class\': per_class,\n              \'note\': \'Detection metrics do not establish exact asset identity or OCR accuracy.\'}\n    Path(args.output).write_text(json.dumps(output, indent=2, allow_nan=False))\n    print(json.dumps(output, indent=2))\n\n\nif __name__ == \'__main__\': main()\n', 'export_backend.py': '"""Export static FP32 raw YOLO11 ONNX plus the manifest required by posify."""\nimport argparse\nimport hashlib\nimport json\nimport shutil\nfrom pathlib import Path\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--weights\', required=True)\n    parser.add_argument(\'--output\', default=\'/content/backend_export\')\n    parser.add_argument(\'--model-id\', default=\'nameplate-detector\')\n    parser.add_argument(\'--revision\', default=\'v1\')\n    parser.add_argument(\'--imgsz\', type=int, default=640)\n    parser.add_argument(\'--reference-image\')\n    args = parser.parse_args()\n    if args.imgsz < 32 or args.imgsz > 2048 or args.imgsz % 32:\n        parser.error(\'Image size must be a multiple of 32 between 32 and 2048\')\n    out = Path(args.output).resolve()\n    if out.exists() and any(out.iterdir()): parser.error(\'Output directory must be empty\')\n    out.mkdir(parents=True, exist_ok=True)\n    import numpy as np\n    import onnx\n    import onnxruntime as ort\n    from PIL import Image\n    from ultralytics import YOLO\n    import ultralytics\n    model = YOLO(args.weights)\n    if model.task != \'detect\': parser.error(\'Only detection checkpoints are supported\')\n    names = model.names\n    classes = [names[i] for i in range(len(names))] if isinstance(names,dict) else list(names)\n    exported = Path(model.export(format=\'onnx\', imgsz=args.imgsz, batch=1, dynamic=False,\n                                  half=False, simplify=False, nms=False, opset=17, device=\'cpu\'))\n    # Consolidate external tensor data into a single portable ONNX file.\n    graph = onnx.load(str(exported), load_external_data=True)\n    onnx.checker.check_model(graph)\n    artifact = out/\'model.onnx\'\n    onnx.save_model(graph, str(artifact), save_as_external_data=False)\n    session = ort.InferenceSession(str(artifact), providers=[\'CPUExecutionProvider\'])\n    inputs, outputs = session.get_inputs(), session.get_outputs()\n    expected = [1,3,args.imgsz,args.imgsz]\n    if len(inputs) != 1 or inputs[0].shape != expected or inputs[0].type != \'tensor(float)\' or len(outputs) != 1:\n        raise RuntimeError(\'Export does not match the backend static float32 input/output contract\')\n    if args.reference_image:\n        with Image.open(args.reference_image) as source:\n            image=source.convert(\'RGB\'); w,h=image.size; ratio=min(args.imgsz/w,args.imgsz/h)\n            nw,nh=round(w*ratio),round(h*ratio)\n            canvas=Image.new(\'RGB\',(args.imgsz,args.imgsz),(114,114,114))\n            canvas.paste(image.resize((nw,nh),Image.Resampling.BILINEAR),((args.imgsz-nw)//2,(args.imgsz-nh)//2))\n            tensor=np.ascontiguousarray(np.asarray(canvas,dtype=np.float32).transpose(2,0,1)[None]/255.)\n    else:\n        tensor=np.zeros(expected,dtype=np.float32)\n    prediction=session.run(None,{inputs[0].name:tensor})[0]\n    if prediction.ndim != 3 or prediction.shape[:2] != (1,4+len(classes)):\n        raise RuntimeError(f\'Unsupported output shape: {prediction.shape}; expected [1,4+C,N] without embedded NMS\')\n    if not np.isfinite(prediction).all() or np.any(prediction[:,4:,:] < 0) or np.any(prediction[:,4:,:] > 1):\n        raise RuntimeError(\'Export produced invalid coordinates or class probabilities\')\n    digest=hashlib.sha256(artifact.read_bytes()).hexdigest()\n    manifest={\'model_id\':args.model_id,\'revision\':args.revision,\'artifact\':\'model.onnx\',\'sha256\':digest,\n              \'format\':\'onnx\',\'output_layout\':\'yolo11_raw_xywh_class_scores\',\'input_size\':[args.imgsz,args.imgsz],\n              \'class_names\':classes,\'confidence_threshold\':.4,\'iou_threshold\':.5}\n    (out/\'manifest.json\').write_text(json.dumps(manifest,indent=2))\n    np.save(out/\'reference_input.npy\',tensor)\n    np.save(out/\'reference_raw_output.npy\',prediction)\n    (out/\'export_report.json\').write_text(json.dumps({\'ultralytics\':ultralytics.__version__,\n        \'input_shape\':expected,\'output_shape\':list(prediction.shape),\'class_names\':classes,\n        \'sha256\':digest,\'reference\':\'supplied_image\' if args.reference_image else \'zeros_conformance_only\'},indent=2))\n    archive=shutil.make_archive(str(out),\'zip\',root_dir=out)\n    print(\'Backend artifact bundle:\',archive)\n    print(\'No accuracy claim is made by a successful ONNX conformance check.\')\n\n\nif __name__ == \'__main__\': main()\n'}
for k, v in FILES.items():
    (ROOT / k).write_text(v, encoding="utf-8")
print("Wrote starter files:", list(FILES.keys()))
%cd /content/posify_ml_starter


In [ ]:
!pip install -q -r requirements-colab.txt
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
!python build_colab_dataset.py --output /content/dataset --train-count 80 --val-count 20 --test-count 20
!python validate_dataset.py --data /content/dataset/data.yaml --report /content/dataset_report.json


In [ ]:
import torch
dev = "0" if torch.cuda.is_available() else "cpu"
!python train_yolo.py --data /content/dataset/data.yaml --weights yolo11n.pt --epochs 25 --batch 16 --device {dev} --name equipment_yolo11n_v2


In [ ]:
import torch
dev = "0" if torch.cuda.is_available() else "cpu"
!python evaluate_yolo.py --weights /content/runs/equipment_yolo11n_v2/weights/best.pt --data /content/dataset/data.yaml --split test --device {dev} --output /content/detector_evaluation.json


In [ ]:
!python export_backend.py --weights /content/runs/equipment_yolo11n_v2/weights/best.pt --output /content/backend_export --model-id equipment-detector --revision v2 --reference-image /content/dataset/test/images/test_0000.jpg
from google.colab import files
files.download("/content/backend_export.zip")
files.download("/content/detector_evaluation.json")
files.download("/content/dataset_report.json")
